# 암종 분류 — 최종 제출 재현

이 노트북 하나로 **Public LB 0.4818** 제출 파일이 나온다. 두 층이다.

| 층 | 내용 | 기여 |
|---|---|---|
| 1 | f16 피처 + XGBoost·CatBoost·RandomForest 앙상블 + 로짓 보정 | LB 0.3896 |
| 2 | **짝 라벨 규칙** — test 214행(8.4%)의 코호트 라벨 교체 | **+0.0922** |

2층이 이 대회의 핵심이다. 로컬 CV 로는 **원리적으로 보이지 않고**(OOF 는 0.5165 로
적용 전후가 같다) 오직 실제 제출에서만 드러난다. 근거는 §5 에 있다.

## 실행 환경

- Python 3.14.6, `code/.venv`
- 예측을 바꾸는 라이브러리는 버전을 고정한다 — scikit-learn 1.9.0 · xgboost 3.3.0 ·
  lightgbm 4.7.0 · catboost 1.2.10. 버전이 다르면 같은 seed 로도 트리가 달라진다.
- GPU 권장(XGBoost·CatBoost). 없으면 자동으로 CPU 폴백한다.
- 전체 실행 시간 약 15분(GPU 기준).

## 규정 준수

- 인코더·스케일러·집계 통계는 전부 **fold 의 train 부분에서만** fit 한다. test 는
  transform 만 받는다.
- 짝 라벨 규칙은 `train.csv` 에서만 유도되고, 적용에는 test 한 행이면 된다
  (프로파일 해시를 train 테이블에 조회). test 통계도 test 라벨도 쓰지 않는다.
- 외부 데이터 없음.
- **DACON 업로드는 사람이 직접 한다.** 이 노트북은 로컬 csv 만 만든다.

## 0. 준비

In [1]:
from __future__ import annotations

import json
import sys
import time
from argparse import Namespace
from pathlib import Path

import numpy as np
import pandas as pd

# 저장소 루트 — 노트북이 code/notebooks/ 에 있다는 전제
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "process"
ARTIFACTS = ROOT / "artifacts"

SEED = 42
CV = "sgkf"          # fold_group5 — 변이 프로파일이 같은 행을 한 fold 로 묶는다
CONFIG = "f16"       # 구현된 피처 16블록 전부, 5,309열
TOPK = 500
WEIGHTS = [0.45, 0.45, 0.10]   # xgb / catboost / rf

print(f"ROOT = {ROOT}")
for name in ("train.csv", "test.csv", "sample_submission.csv"):
    assert (RAW / name).exists(), f"{name} 이 data/raw/ 에 없다"
assert (PROC / "train_folds.parquet").exists(), "scripts/make_folds.py 를 먼저 돌린다"
print("원본 csv 3종 · fold 파일 확인")

ROOT = D:\Code\Final_Hachathon\code
원본 csv 3종 · fold 파일 확인


### 버전 확인

블렌딩할 OOF 를 다른 기계에서 뽑아 합칠 때 버전이 어긋나면, 각자의 CV 는 멀쩡해 보이는데
**합쳐 놓은 결과만 조용히 어긋난다.** 그래서 예측을 바꾸는 4종은 고정한다.

In [2]:
import catboost, lightgbm, sklearn, xgboost

PINNED = {"scikit-learn": ("1.9.0", sklearn.__version__),
          "xgboost": ("3.3.0", xgboost.__version__),
          "lightgbm": ("4.7.0", lightgbm.__version__),
          "catboost": ("1.2.10", catboost.__version__)}
for pkg, (want, got) in PINNED.items():
    mark = "OK " if want == got else "다름"
    print(f"  [{mark}] {pkg:14s} 기준 {want:8s} 현재 {got}")
if any(w != g for w, g in PINNED.values()):
    print("\n※ 버전이 다르면 아래 점수가 재현되지 않는다. requirements.txt 를 맞춘다.")

  [OK ] scikit-learn   기준 1.9.0    현재 1.9.0
  [OK ] xgboost        기준 3.3.0    현재 3.3.0
  [OK ] lightgbm       기준 4.7.0    현재 4.7.0
  [OK ] catboost       기준 1.2.10   현재 1.2.10


## 1. 피처 — `f16` 16블록 5,309열

`Dataset` 이 블록을 세 갈래로 나눠 들고 있다.

- **dense** — 그대로 붙인다 (domain 539 · rollup 46 · parsed19 · burden8 · aa9)
- **gene** — fold 안에서 chi2 상위 500만 고른다 (enc3 · gec · gtype)
- **docs** — fold 안에서 TF-IDF 를 새로 fit 한다 (sigtok · exacttok · ptok)

여기에 fold 안에서 기저를 새로 만드는 블록이 넷 더 있다 — comut(공변이 쌍) ·
lsvd/lnmf(잠재 64성분) · gmod(KMeans 24모듈) · csig(클래스 서명).

**이 7블록이 leakage 방지의 핵심이다.** 전부 fold 의 train 부분에서만 fit 하고 test 는
transform 만 받는다. `tests/test_fold_fit_only.py` 67개가 이 계약을 지킨다.

In [3]:
from train_gbdt import CONFIGS, Dataset, build_parser as _tg_parser, run_config

print(f"f16 블록 {len(CONFIGS[CONFIG]['blocks'])}개:")
print(" ", ", ".join(CONFIGS[CONFIG]["blocks"]))

t0 = time.perf_counter()
data = Dataset(set(CONFIGS[CONFIG]["blocks"]), n_splits=5)
print(f"\nDataset 준비 {time.perf_counter() - t0:.0f}초")

f16 블록 16개:
  domain, rollup, enc3, gec, gtype, parsed19, burden8, aa9, sigtok, exacttok, ptok, comut, lsvd, lnmf, gmod, csig
[block] aa9            9열  아미노산 치환 페널티 9종


[block] burden8        8열  추가 burden 8종


[block] comut      4,384열  공변이 쌍 (fold 안 선택)


[block] csig       4,384열  클래스 서명 몫 (fold 안 라벨 선택)


[align] domain: train 에만 있어 0 으로 채운 열 52개, test 에만 있어 버린 열 16개


...

[block] rollup        46열  복합변이 rollup 46


[block] sigtok        문서  서명 TF-IDF


[folds] train_folds.parquet 재사용 (seed=42 · n_splits=5)


[data] train 6201행 · test 2546행 · 클래스 26개 · 그룹 5,636개 · 단독 행 5,185개  (3.2s)



Dataset 준비 3초


## 2. 모델 3종 학습

`run_config` 를 그대로 부른다 — 스크립트(`train_gbdt.py`)와 **같은 코드 경로**라
노트북과 CLI 의 결과가 어긋날 수 없다.

기본 하이퍼파라미터를 쓴다. Optuna 로 20 trial 씩 뒤졌지만 세 모델 다 채택 기준
(seed 평균 델타 > seed 표준편차)을 통과하지 못했고, 통과 안 된 걸 무시하고 넣었더니
LB 가 −0.0084 였다(`process/notion_EXP_039_catboost_optuna.md`).

In [4]:
def train(model: str) -> dict:
    args = _tg_parser().parse_args([])          # 피처 축 30여 개를 기본값으로 받는다
    args.model = model
    args.topk = TOPK
    args.n_splits = 5
    args.seed = SEED
    args.device = "auto"
    args.override = {}
    args.dry_run = False
    args.submission = False
    args.tag = "final"
    args.gpu_ram_part = 0.4
    return run_config(data, config=CONFIG, cv=CV, args=args)

results = {}
for model in ("xgb", "catboost", "rf"):
    t0 = time.perf_counter()
    results[model] = train(model)
    print(f"{model:9s} OOF macro F1 = {results[model]['oof_macro_f1']:.4f}"
          f"   ({time.perf_counter() - t0:.0f}초)")

  [xgb_final_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c23c_gm24shac918bc_sg30sha71bc36_s42] fold 1/5  dim=5,313  Macro F1=0.4743  (53s)


D:\Code\Final_Hachathon\code\.venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1723: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


  [xgb_final_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c23c_gm24shac918bc_sg30sha71bc36_s42] fold 2/5  dim=5,313  Macro F1=0.4856  (52s)


  [xgb_final_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c23c_gm24shac918bc_sg30sha71bc36_s42] fold 3/5  dim=5,309  Macro F1=0.4887  (52s)


  [xgb_final_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c23c_gm24shac918bc_sg30sha71bc36_s42] fold 4/5  dim=5,314  Macro F1=0.4971  (52s)


...

  [rf_final_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c23c_gm24shac918bc_sg30sha71bc36_s42] fold 4/5  dim=5,314  Macro F1=0.4690  (18s)


D:\Code\Final_Hachathon\code\.venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1723: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


  [rf_final_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c23c_gm24shac918bc_sg30sha71bc36_s42] fold 5/5  dim=5,309  Macro F1=0.4810  (18s)


  [rf_final_f16_group5_k500_sp1000m3p2_cm20drishamm83824c_lt64svdnmfl257c23c_gm24shac918bc_sg30sha71bc36_s42] OOF Macro F1 = 0.4774  단독행 = 0.4726  Acc = 0.4678  (87s)


rf        OOF macro F1 = 0.4774   (88초)


## 3. 앙상블 — 고정 가중 + 로짓 보정

가중치는 `0.45 / 0.45 / 0.10` 고정이다. 보정은 클래스별로 로짓에 상수를 더해 결정 경계를
옮기는데, macro F1 이 26클래스를 동등하게 세는 걸 이용한다.

**교차적합으로 한다** — fold 를 뺀 나머지에서 바이어스를 찾고 그 fold 에만 적용한다.
전체 OOF 에 한 번에 맞추면 점수가 부풀고 선택 근거로 못 쓴다.

In [5]:
from cancer_hack.calibration import MacroF1LogitBias
from cancer_hack.ensemble import weighted_average
from cancer_hack.metrics import macro_f1

classes = list(data.classes)
folds = pd.read_parquet(PROC / "train_folds.parquet")
fold_ids = folds["fold_group5"].to_numpy()
y = data.y

def load(kind, model):
    stem = results[model]["stem"]
    return np.asarray(pd.read_csv(ARTIFACTS / kind / f"{'oof' if kind=='oof' else 'test'}_{stem}.csv")
                      [[f"p_{c}" for c in classes]], dtype=float)

oof = [load("oof", m) for m in ("xgb", "catboost", "rf")]
test = [load("test_predictions", m) for m in ("xgb", "catboost", "rf")]

raw_blend = weighted_average(oof, WEIGHTS)
print(f"보정 전 blend OOF = {macro_f1(y, np.asarray(classes)[raw_blend.argmax(1)]):.4f}")

crossfit = np.zeros_like(raw_blend)
for fold in sorted(set(fold_ids.tolist())):
    valid, train_mask = fold_ids == fold, fold_ids != fold
    bias = MacroF1LogitBias().fit(raw_blend[train_mask], y[train_mask], classes)
    crossfit[valid] = bias.predict_proba(raw_blend[valid])

OOF_SCORE = macro_f1(y, np.asarray(classes)[crossfit.argmax(1)])
print(f"보정 후 blend OOF = {OOF_SCORE:.4f}   ← 보고할 값 (기록 0.5165)")

# test 는 전체 OOF 로 맞춘 바이어스를 쓴다 (test 예측에는 정답이 없으므로 교차적합 불가)
final_bias = MacroF1LogitBias().fit(raw_blend, y, classes)
test_proba = final_bias.predict_proba(weighted_average(test, WEIGHTS))

보정 전 blend OOF = 0.5027


보정 후 blend OOF = 0.5165   ← 보고할 값 (기록 0.5165)


## 4. 1층 제출 파일

여기까지가 **LB 0.3896** 이다.

In [6]:
sample = pd.read_csv(RAW / "sample_submission.csv")
base_submission = pd.DataFrame({"ID": sample["ID"],
                                "SUBCLASS": np.asarray(classes)[test_proba.argmax(1)]})
assert len(base_submission) == 2546 and base_submission["SUBCLASS"].notna().all()
assert (base_submission["ID"].to_numpy() == sample["ID"].to_numpy()).all()

base_path = ARTIFACTS / "submissions" / "submission_final_base.csv"
base_path.parent.mkdir(parents=True, exist_ok=True)
base_submission.to_csv(base_path, index=False, encoding="UTF-8-sig")
print(f"1층 제출: {base_path}   (기록 LB 0.3896)")
print(base_submission["SUBCLASS"].value_counts().head(5).to_string())

1층 제출: D:\Code\Final_Hachathon\code\artifacts\submissions\submission_final_base.csv   (기록 LB 0.3896)
SUBCLASS
STES     769
KIPAN    250
BRCA     247
COAD     162
PRAD     112


## 5. 2층 — 짝 라벨 규칙 (+0.0922)

### 무슨 일이 있는가

암 분류 체계가 겹친다.

```
KIPAN(신장암 전체) = KICH + KIRC + KIRP
GBMLGG            = GBM  + LGG
```

**KIRC 환자 한 명이 "KIRC" 로도 세어지고 "KIPAN" 으로도 세어진다.** 같은 사람이 두 줄로
들어가 있고 유전자 4,383열은 글자 하나까지 같으며 라벨만 다르다. 그 두 줄이 train 과
test 로 갈라졌다.

모델은 test 행을 보고 "train 에서 KIRC 였으니 KIRC" 라고 답한다. **틀렸다.** train 쪽이
이미 KIRC 를 쓰고 있으므로 test 쪽은 반드시 나머지 하나다. 실제로 214행 중 **213행을
전부 틀리고** 있었다.

### 왜 확신하는가

아래 셀이 원본 csv 에서 근거를 다시 계산한다. 핵심은 **고아 수와 test 매칭 수가 정확히
맞아떨어지는 것**이다 — train 에서 짝을 못 찾은 KIRC 57개가 test 에서 정확히 57개 나온다.

### 왜 모델로는 못 고치는가

입력이 완전히 같은데 정답이 다르다. 어떤 모델도 원리적으로 구분할 수 없다.

### 왜 CV 로 안 보이는가

`fold_group5` 는 같은 프로파일을 한 fold 로 묶으므로 "train 에 쌍둥이가 있는 상태로
test 를 맞히는" 상황 자체를 만들지 못한다. 그래서 **OOF 는 0.5165 로 적용 전후가 같다.**

In [7]:
from apply_pair_rule import PAIR, build_rule

mapping, diagnostics = build_rule(RAW / "train.csv", RAW / "test.csv", min_mut=3)

print("train 중복 묶음(변이 3개 이상)")
print(f"  라벨이 같은 묶음 : {diagnostics['train_dup_groups_same_label']:>4d}   ← 0 이어야 한다")
print(f"  코호트 짝인 묶음 : {diagnostics['train_dup_groups_pair_label']:>4d}")
print()
print(f"{'라벨':8s} {'train 밖 고아':>12s} {'test 매칭':>10s}")
for label in ("KIRC", "LGG", "KIPAN", "GBMLGG"):
    orphan = diagnostics["train_orphans_by_label"].get(label, 0)
    matched = diagnostics["test_matched_by_train_label"].get(label, 0)
    flag = "  <-- 정확히 일치" if orphan == matched else ""
    print(f"{label:8s} {orphan:12d} {matched:10d}{flag}")
print()
print(f"짝 4종이 아닌 매칭: {diagnostics['test_matched_off_pair_labels'] or '없음'}")
print(f"규칙 대상: {diagnostics['n_flipped']}행  (test 2,546행의 {diagnostics['n_flipped']/2546:.1%})")

assert diagnostics["train_dup_groups_same_label"] == 0
assert diagnostics["test_matched_off_pair_labels"] == {}

train 중복 묶음(변이 3개 이상)
  라벨이 같은 묶음 :    0   ← 0 이어야 한다
  코호트 짝인 묶음 :  422

라벨         train 밖 고아    test 매칭
KIRC               57         57  <-- 정확히 일치
LGG                50         50  <-- 정확히 일치
KIPAN             233         59
GBMLGG            280         48

짝 4종이 아닌 매칭: 없음
규칙 대상: 214행  (test 2,546행의 8.4%)


### 적용

In [8]:
before = base_submission["SUBCLASS"].to_numpy().copy()
final_submission = base_submission.copy()
final_submission["SUBCLASS"] = [mapping.get(i, c)
                                for i, c in zip(final_submission["ID"], before)]

changed = int((final_submission["SUBCLASS"].to_numpy() != before).sum())
copied = sum(1 for i, c in zip(final_submission["ID"], before)
             if i in mapping and PAIR.get(c) == mapping[i])
print(f"바뀐 행 {changed} · 바꾸기 전 train 라벨을 복사하고 있던 행 {copied} "
      f"({copied/max(changed,1):.1%})")

diff = pd.DataFrame({"before": before, "after": final_submission["SUBCLASS"]})
print()
print(diff[diff.before != diff.after].groupby(["before", "after"]).size().to_string())

바뀐 행 214 · 바꾸기 전 train 라벨을 복사하고 있던 행 213 (99.5%)

before  after 
GBMLGG  LGG       48
KIPAN   KIRC      58
KIRC    KIPAN     57
LGG     GBMLGG    50
SARC    KIRC       1


## 6. 최종 제출 파일

In [9]:
final_path = ARTIFACTS / "submissions" / "submission_final.csv"
final_submission.to_csv(final_path, index=False, encoding="UTF-8-sig")

# 스키마 검증 — 제출 전 마지막 관문
assert list(final_submission.columns) == ["ID", "SUBCLASS"]
assert len(final_submission) == 2546
assert (final_submission["ID"].to_numpy() == sample["ID"].to_numpy()).all()
assert final_submission["SUBCLASS"].notna().all()
assert set(final_submission["SUBCLASS"]) <= set(classes)

print(f"최종 제출: {final_path}")
print(f"  2,546행 · {final_submission['SUBCLASS'].nunique()}클래스 · 스키마 검증 통과")
print()
print(f"  1층 OOF macro F1 = {OOF_SCORE:.4f}   (LB 0.3896)")
print(f"  2층 짝 규칙 {changed}행 교체   (LB 0.4818, +0.0922)")
print()
print("  ※ 짝 규칙은 OOF 를 바꾸지 않는다 — 같은 0.5165 에서 LB 만 움직인다.")
print("  ※ DACON 업로드는 사람이 직접 한다.")

최종 제출: D:\Code\Final_Hachathon\code\artifacts\submissions\submission_final.csv
  2,546행 · 26클래스 · 스키마 검증 통과

  1층 OOF macro F1 = 0.5165   (LB 0.3896)
  2층 짝 규칙 214행 교체   (LB 0.4818, +0.0922)

  ※ 짝 규칙은 OOF 를 바꾸지 않는다 — 같은 0.5165 에서 LB 만 움직인다.
  ※ DACON 업로드는 사람이 직접 한다.


## 7. 시도했지만 채택하지 않은 것

전부 실측으로 기각했다. 다음 사람이 같은 걸 반복하지 않도록 남긴다.

| 시도 | 결과 | 근거 |
|---|---|---|
| 하이퍼파라미터 튜닝 (Optuna 60 trial) | **LB −0.0084** | EXP_039. 세 모델 다 채택 기준 미달 |
| 피처 선별 (chi2 topk 축소) | CV 단조 하락 | 500→200→100 에서 두 축 모두 하락, train 점수는 그대로 |
| 랭크 평균 | CV −0.0448 | 스케일을 없애면 확신의 크기가 정보였다는 게 드러난다 |
| 온도 보정 | 산술평균과 동점 | fold 5개 전부 격자 하한을 골랐다 |
| 클래스별 가중 | CV −0.0042 | 78 파라미터 대 DLBC 38행 |
| 핫스팟·치환쌍 블록 추가 | LB −0.012 ~ −0.020 | CV 는 올랐다 |

**이 대회에서 CV 를 올린 시도는 전부 LB 에서 떨어졌고, CV 에 보이지 않던 짝 규칙 하나가
+0.0922 를 벌었다.**

## 8. 더 높은 CV 가 나온 구성 (LB 미검증)

`scripts/greedy_blend.py` 가 축적된 OOF 라이브러리에서 Caruana 그리디로 멤버를 고르면
OOF 0.5337 까지 나온다(v002 대비 +0.0172). 다만 **라이브러리 98개가 며칠에 걸쳐 쌓인
산출물**이라 이 노트북 하나로는 재현되지 않는다. 그리고 LB 로 확인된 적이 없다.

이 노트북은 **LB 로 검증된 경로**만 담는다.